In [ ]:
# Cell 1 - Startup check
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import subprocess
import sys
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import yaml

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "experiments/cgan_v1/runs/2026-06-05_smoke/config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

legacy_code_dir = Path(config["data"]["legacy_code_dir"])
if str(legacy_code_dir) not in sys.path:
    sys.path.insert(0, str(legacy_code_dir))

from rf_models import TinyResidualRFNet
from rf_cached_dataset import RFCachedDataset
from rf_cgan_models import Envelope2DPatchDiscriminator, count_trainable_params, extract_envelope_slice

def git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception as exc:
        return f"UNKNOWN ({exc})"

assert config["generator"]["init"] == "random", "Smoke test must cold-start G; do not load regression checkpoints."
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

G = TinyResidualRFNet(use_batch_norm=bool(config["generator"]["use_bn"]))
D = Envelope2DPatchDiscriminator(ndf=int(config["discriminator"]["ndf"]))

print(f"{datetime.now().isoformat(timespec='seconds')} | TASK: cGAN smoke test startup")
print(f"run_name: {config['run_name']}")
print(f"git_commit: {git_commit()}")
print(f"project_root: {PROJECT_ROOT}")
print(f"legacy_code_dir: {legacy_code_dir}")
print(f"device: {device}")
print("config:")
print(yaml.safe_dump(config, sort_keys=False))
print(f"G: {type(G).__name__}, trainable_params={count_trainable_params(G):,}")
print(f"D: {type(D).__name__}, trainable_params={count_trainable_params(D):,}")


In [ ]:
# Cell 2 - Imports and config verification
required_top_keys = {"run_name", "data", "generator", "discriminator", "loss", "training", "logging"}
missing = required_top_keys.difference(config)
assert not missing, f"Missing config keys: {sorted(missing)}"
assert config["loss"]["fidelity_type"] == "complex_l1"
assert config["loss"]["type"] == "lsgan"
assert config["training"]["num_workers"] == 0
assert config["status"] == "smoke_test"
print("PASS: imports and config verification complete")


In [ ]:
# Cell 3 - Dataset and DataLoader
cache_root = Path(config["data"]["cache_dir"])
split_cache = cache_root / config["data"]["split"]
assert split_cache.exists(), f"Missing cache split: {split_cache}"

dataset = RFCachedDataset(split_cache)
max_samples = min(int(config["data"]["max_samples"]), len(dataset))
subset = Subset(dataset, list(range(max_samples)))
loader = DataLoader(
    subset,
    batch_size=int(config["training"]["batch_size"]),
    shuffle=True,
    num_workers=int(config["training"]["num_workers"]),
    pin_memory=torch.cuda.is_available(),
)

sample = dataset[0]
print(f"dataset: {split_cache}")
print(f"full_samples={len(dataset)}, smoke_samples={len(subset)}, batches_per_epoch={len(loader)}")
print(f"sample input={tuple(sample['input'].shape)}, label={tuple(sample['label'].shape)}, baseline={tuple(sample['baseline'].shape)}")


In [ ]:
# Cell 4 - Model and optimizer initialization
G = G.to(device)
D = D.to(device)

betas = (float(config["training"]["beta1"]), float(config["training"]["beta2"]))
opt_G = torch.optim.Adam(G.parameters(), lr=float(config["training"]["lr_G"]), betas=betas)
opt_D = torch.optim.Adam(D.parameters(), lr=float(config["training"]["lr_D"]), betas=betas)
criterion = nn.MSELoss()
use_amp = bool(config["training"]["use_amp"]) and torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print(f"PASS: model/optimizer init complete | use_amp={use_amp}")
print(f"G params={count_trainable_params(G):,}, D params={count_trainable_params(D):,}")


In [ ]:
# Cell 5 - Smoke training loop (20 epochs)
epochs = int(config["training"]["epochs"])
lambda_fid = float(config["loss"]["lambda_fidelity"])
snapshot_epochs = set(int(x) for x in config["logging"]["snapshot_epochs"])
abort_if_d_fake_lt = float(config["logging"]["abort_if_d_fake_lt"])

history = []
abort_reason = None

for epoch in range(1, epochs + 1):
    t0 = time.time()
    G.train()
    D.train()
    sums = {"d_real": 0.0, "d_fake": 0.0, "d_loss": 0.0, "g_adv": 0.0, "g_fid": 0.0, "g_loss": 0.0}
    n_batches = 0

    for batch in loader:
        x = batch["input"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        baseline = batch["baseline"].to(device, non_blocking=True)

        # D update
        opt_D.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = G(x, baseline)
            label_env = extract_envelope_slice(label)
            baseline_env = extract_envelope_slice(baseline)
            pred_env = extract_envelope_slice(pred.detach())

            d_real_out = D(torch.cat([label_env, baseline_env], dim=1))
            d_fake_out = D(torch.cat([pred_env, baseline_env], dim=1))
            d_real_loss = criterion(d_real_out, torch.ones_like(d_real_out))
            d_fake_loss = criterion(d_fake_out, torch.zeros_like(d_fake_out))
            d_loss = 0.5 * (d_real_loss + d_fake_loss)

        scaler.scale(d_loss).backward()
        scaler.step(opt_D)

        # G update
        opt_G.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = G(x, baseline)
            pred_env = extract_envelope_slice(pred)
            baseline_env = extract_envelope_slice(baseline)
            d_for_g = D(torch.cat([pred_env, baseline_env], dim=1))
            g_adv = criterion(d_for_g, torch.ones_like(d_for_g))
            g_fid = torch.mean(torch.abs(pred - label))
            g_loss = g_adv + lambda_fid * g_fid

        scaler.scale(g_loss).backward()
        scaler.step(opt_G)
        scaler.update()

        with torch.no_grad():
            d_real_score = torch.sigmoid(d_real_out.float()).mean().item()
            d_fake_score = torch.sigmoid(d_fake_out.float()).mean().item()

        values = {
            "d_real": d_real_score,
            "d_fake": d_fake_score,
            "d_loss": float(d_loss.detach().float().item()),
            "g_adv": float(g_adv.detach().float().item()),
            "g_fid": float(g_fid.detach().float().item()),
            "g_loss": float(g_loss.detach().float().item()),
        }
        if any(torch.isnan(torch.tensor(list(values.values())))):
            abort_reason = f"NaN at epoch {epoch}, batch {n_batches + 1}: {values}"
            break
        if values["d_fake"] < abort_if_d_fake_lt:
            abort_reason = f"D winning completely at epoch {epoch}, batch {n_batches + 1}: d_fake={values['d_fake']:.6f}"
            break

        for key in sums:
            sums[key] += values[key]
        n_batches += 1

    if n_batches > 0:
        row = {key: sums[key] / n_batches for key in sums}
    else:
        row = {key: float("nan") for key in sums}
    row["epoch"] = epoch
    row["seconds"] = time.time() - t0
    history.append(row)

    print(
        f"epoch {epoch:03d} | d_real={row['d_real']:.4f} d_fake={row['d_fake']:.4f} "
        f"d_loss={row['d_loss']:.4f} g_adv={row['g_adv']:.4f} "
        f"g_fid={row['g_fid']:.6f} g_loss={row['g_loss']:.4f} sec={row['seconds']:.1f}"
    )
    if epoch in snapshot_epochs:
        print(
            f"SNAPSHOT epoch={epoch} D_real={row['d_real']:.6f} D_fake={row['d_fake']:.6f} "
            f"G_adv={row['g_adv']:.6f} G_fidelity={row['g_fid']:.6f}"
        )
    if abort_reason is not None:
        print(f"EARLY_ABORT: {abort_reason}")
        break


In [ ]:
# Cell 6 - Result summary
run_dir = PROJECT_ROOT / "experiments/cgan_v1/runs/2026-06-05_smoke"
log_dir = run_dir / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

print("epoch,d_real,d_fake,d_loss,g_adv,g_fid,g_loss,seconds")
for row in history:
    print(
        f"{row['epoch']},{row['d_real']:.6f},{row['d_fake']:.6f},{row['d_loss']:.6f},"
        f"{row['g_adv']:.6f},{row['g_fid']:.9f},{row['g_loss']:.6f},{row['seconds']:.3f}"
    )

completed = (len(history) == int(config["training"]["epochs"])) and abort_reason is None
final_d_fake = history[-1]["d_fake"] if history else float("nan")
g_adv_values = [row["g_adv"] for row in history]
d_fake_values = [row["d_fake"] for row in history]
nontrivial_movement = (
    len(history) >= 2
    and (max(g_adv_values) - min(g_adv_values) > 1e-4)
    and (max(d_fake_values) - min(d_fake_values) > 1e-4)
)
if completed and 0.1 <= final_d_fake <= 0.9 and nontrivial_movement:
    status = "PASS"
elif abort_reason:
    status = "EARLY_ABORT"
else:
    status = "INCONCLUSIVE"

print(f"SMOKE_STATUS: {status}")
if abort_reason:
    print(f"SMOKE_ABORT_REASON: {abort_reason}")

summary_path = log_dir / "smoke_summary.txt"
with summary_path.open("w", encoding="utf-8") as f:
    f.write(f"run_name: {config['run_name']}\n")
    f.write(f"status: {status}\n")
    f.write(f"abort_reason: {abort_reason}\n")
    f.write("epoch,d_real,d_fake,d_loss,g_adv,g_fid,g_loss,seconds\n")
    for row in history:
        f.write(
            f"{row['epoch']},{row['d_real']:.6f},{row['d_fake']:.6f},{row['d_loss']:.6f},"
            f"{row['g_adv']:.6f},{row['g_fid']:.9f},{row['g_loss']:.6f},{row['seconds']:.3f}\n"
        )
print(f"wrote {summary_path}")
